In [3]:
%pip install viennarna

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 2.0 MB/s  0:00:03 eta 0:00:010m
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for viennarna: filename=viennarna-2.7.2-cp39-cp39-macosx_10_9_universal2.whl size=5083343 sha256=5e1c60b13ffd2379faebf361fa62598ff3615798941ce947d911ab811d0808aa
  Stored in directory: /Users/beck049/Library/Caches/pip/wheels/ee/55/f1/228d80f26a999141390d1e88203db2458a29bf01dd5af00bfb
Successfully built viennarna
Note: you may need to restart the kernel to use updated packages.


# ViennaRNA
input : codon sequence
output: 2D structure & the corresponding MFE

## Object Function
McCaskill 演算法 (Partition Function & Ensemble)
除了尋找單一的最穩定結構（MFE），ViennaRNA 還透過 McCaskill 演算法 計算系統的配分函數（Partition Function, $Q$），這時的目標函數擴展到整個熱力學平衡系綜（Ensemble）：
$$Q = \sum_{S \in \mathcal{S}(x)} e^{-\frac{\Delta G(x, S)}{R T}}$$

- $R$：氣體常數。
- $T$：絕對溫度（預設為 37 °C / 310.15 K）。

透過配分函數 $Q$，ViennaRNA 可以計算出：
- 系綜自由能（Ensemble Free Energy）： $G_{\text{ensemble}} = -R T \ln(Q)$
- 特定結構 $S$ 出現的玻爾茲曼機率： $P(S) = \frac{e^{-\frac{\Delta G(x, S)}{R T}}}{Q}$
- 鹼基對配對機率（Base Pairing Probabilities）：計算任意兩個鹼基 $i$ 與 $j$ 形成配對的總機率。

## Compare

| | Vienna | LCDSfold |
|---|---|---|
|Object Function| MFE | MFE + CAI |
|MFE Model|McCaskill 演算法 (Partition Function & Ensemble)|Loop-based 熱力學模型|

In [ ]:
import RNA
import time

# read data from the output of LCDSfold
result = """
Circular RNA Reconstruction:
Coding sequence and its secondary structure :
AUGGGUCAGAGCAAGUCCAAGGAGGAGAAGGGCAUCAGCGGCACCAGUCGGGCAGAGAUUCUGCCCGACACCACCUACCUGGGGCCGCUGAAUUGCAAGAGCUGUUGGCAGAAGUUCGACAGCUUUAGCAAGUGCCACGACCACUACCUGUGUCGGCACUGCCUGAAUCUCCUCCUGACUAGCUCUGACCGGUGCCCCCUGUGCAAGUACCCCCUG
((.(((((((((.((((..(((((((((.(((((((((((((.(((((((((((((...))))))))))..........))).)))))))).((((.((((((((((((....)).)))))))))).))))(((((..((.(((.....)))))))))))))))...))))))))))))).))))))))).)).......................
Folding free energy: -116.7 kcal/mol
CAI: 0.912289
Total runtime: 0.002 s
"""

result_lines = [line.strip() for line in result.strip().splitlines()]
sequence = result_lines[2]

start_time = time.perf_counter()

# 1. 建立 Model Details 物件並指定為環狀 RNA
md = RNA.md()
md.circ = 1  # 啟用環狀 RNA (circRNA) 計算模式

# 2. 建立 fold_compound 物件
fc = RNA.fold_compound(sequence, md)

# 3. 計算 MFE 結構與自由能
structure, mfe = fc.mfe()

end_time = time.perf_counter()
execution_time = end_time - start_time


print("====== LCDSfold ======")
print("Coding sequence and its secondary structure :")
print(sequence)
print(result_lines[3])
print(result_lines[4])
print(result_lines[6])

print("======  Vienna  ======")
print("Coding sequence and its secondary structure :")
print(sequence)
print(structure)
print(f"Folding free energy: {mfe:.2f} kcal/mol")
print(f"Total runtime: {execution_time:.3f} s")

====== LCDSfold ======
Coding sequence and its secondary structure :
AUGGGUCAGAGCAAGUCCAAGGAGGAGAAGGGCAUCAGCGGCACCAGUCGGGCAGAGAUUCUGCCCGACACCACCUACCUGGGGCCGCUGAAUUGCAAGAGCUGUUGGCAGAAGUUCGACAGCUUUAGCAAGUGCCACGACCACUACCUGUGUCGGCACUGCCUGAAUCUCCUCCUGACUAGCUCUGACCGGUGCCCCCUGUGCAAGUACCCCCUG
((.(((((((((.((((..(((((((((.(((((((((((((.(((((((((((((...))))))))))..........))).)))))))).((((.((((((((((((....)).)))))))))).))))(((((..((.(((.....)))))))))))))))...))))))))))))).))))))))).)).......................
Folding free energy: -116.7 kcal/mol
Total runtime: 0.002 s
======  Vienna  ======
Coding sequence and its secondary structure :
AUGGGUCAGAGCAAGUCCAAGGAGGAGAAGGGCAUCAGCGGCACCAGUCGGGCAGAGAUUCUGCCCGACACCACCUACCUGGGGCCGCUGAAUUGCAAGAGCUGUUGGCAGAAGUUCGACAGCUUUAGCAAGUGCCACGACCACUACCUGUGUCGGCACUGCCUGAAUCUCCUCCUGACUAGCUCUGACCGGUGCCCCCUGUGCAAGUACCCCCUG
...(((((((((.((((..(((((((((.((((.((((((((.(((((((((((((...))))))))))..........))).))))))))..(((.((((((((((((....).))))))))))).)))((((((..(((.((.....))